In [1]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt"
)

('input.txt', <http.client.HTTPMessage at 0x7b7da31ddaf0>)

## Setup & Data Loading

Notes from KV cache implementation:
- During inference (after prefill), only one new token arrives each step
- Q is computed from only the new token (size 1)
- K and V can be concatenated onto whatever you already cached
- Q @ K^T still works if Q has shape (B, 1, hs) and K has shape (B, T, hs) → result is (B, 1, T)
- We don't need the causal mask during decode because we are only considering the most recent token

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import time
import heapq
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from dataclasses import dataclass

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss, _ = model(X, Y)  # unpack 3 return values now
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [3]:
@dataclass
class Request:
    """Each in-flight generation carries its own state and KV cache."""
    id: int
    prompt_tokens: List[int]          # the original encoded prompt
    max_new_tokens: int               # how many tokens this request wants
    generated_tokens: List[int] = field(default_factory=list)
    status: str = "waiting"           # "waiting" -> "prefilling" -> "active" -> "done"
    prefill_cursor: int = 0
    priority: int = 0         # 0 = highest priority
    arrival_time: int = 0     # set when admitted to the scheduler

    # Hint 2: Per-request KV cache, keyed by (layer_idx, head_idx)
    # Each value is a (key_tensor, value_tensor) tuple of shape (1, T_i, head_size)
    # T_i grows by 1 each decode step — different requests have different T_i
    kv_cache: Dict[Tuple[int, int], Tuple[torch.Tensor, torch.Tensor]] = field(
        default_factory=dict
    )

    @property
    def tokens_so_far(self) -> List[int]:
        """Full sequence: prompt + everything generated."""
        return self.prompt_tokens + self.generated_tokens

    @property
    def num_generated(self) -> int:
        return len(self.generated_tokens)

    @property
    def is_done(self) -> bool:
        return self.num_generated >= self.max_new_tokens
    
    @property
    def is_fully_prefilled(self) -> bool:
        return self.prefill_cursor == len(self.prompt_tokens)

    def clear_cache(self):
        self.kv_cache.clear()

In [ ]:
class Scheduler:
    def __init__(self, policy="fcfs", max_batch_size=4, token_budget=16, max_kv_tokens=22):
        self.policy = policy
        self.max_batch_size = max_batch_size
        self.token_budget = token_budget
        self.max_kv_tokens = max_kv_tokens

        self.waiting = []
        self.prefilling = []
        self.active = []
        self.preempted = []

    def promote(self, req):
        self.prefilling.remove(req)
        req.status = "active"
        self.active.append(req)
    
    def complete(self, req):
        self.active.remove(req)
        req.status = "done"

    def _sort_key(self, req):
        if self.policy == "fcfs":
            return (0, req.arrival_time)
        elif self.policy == "priority":
            return (req.priority, req.arrival_time)
    
    def add_request(self, req):
        key = self._sort_key(req)
        heapq.heappush(self.waiting, (*key, req.id, req))
    
    def is_done(self):
        return not (self.waiting or self.prefilling or self.active)
    
    def _maybe_admit(self, step):
        if self.prefilling or not self.waiting:
            return
        
        if len(self.active) >= self.max_batch_size:
            return

        candidate = self.waiting[0][-1]  # or: _, _, _, candidate = self.waiting[0]
        kv_used = sum(len(r.prompt_tokens) + r.num_generated for r in self.active)
        if kv_used + len(candidate.prompt_tokens) > self.max_kv_tokens:
            return
        
        heapq.heappop(self.waiting)
        candidate.arrival_time = step
        candidate.status = "prefilling"
        self.prefilling.append(candidate)
    
    def _maybe_preempt(self):
        kv_used = sum(len(req.prompt_tokens) + req.num_generated for req in self.active + self.prefilling)

        while self.active and kv_used > self.max_kv_tokens:
            victim = max(self.active, key=lambda r: (r.priority, -r.arrival_time))
            self.active.remove(victim)
            victim.clear_cache()
            victim.prefill_cursor = 0
            victim.generated_tokens = []
            victim.status = "waiting"
            self.preempted.append(victim)

            key = self._sort_key(victim)
            heapq.heappush(self.waiting, (*key, victim.id, victim))
            kv_used = sum(len(req.prompt_tokens) + req.num_generated for req in self.active + self.prefilling)

    def schedule(self, step: int):
        """
        Returns:
            prefill_req:  Request | None  — one request getting a prefill chunk (or None)
            decode_reqs:  List[Request]   — all requests currently being decoded (active)

        """
        self._maybe_admit(step)       # promote waiting → prefilling if memory allows
        self._maybe_preempt()         # evict if over memory budget

        prefill_req = self.prefilling[0] if self.prefilling else None
        decode_reqs = list(self.active)

        return prefill_req, decode_reqs


## Hint 2: Stateless Head — KV Cache Moved Outside the Model

**Before:** `Head` owned `self.key_cache` and `self.value_cache` — one monolithic `(B, T, hs)` tensor.
This breaks when different requests have different sequence lengths.

**After:** `Head` is stateless. The cache is:
1. Passed **into** `forward()` as `past_k, past_v`
2. Returned **out of** `forward()` as updated `(new_k, new_v)`
3. **Stored on the `Request` object**, keyed by `(layer_idx, head_idx)`

This threads through: `Head` → `MultiHeadAttention` → `Block` → `GPTLanguageModel`.

Also changed `nn.Sequential` → `nn.ModuleList` for `self.blocks` so we can pass
per-block cache into each block individually.

In [5]:
class Head(nn.Module):
    """One head of self-attention — now STATELESS (no internal cache)."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_k=None, past_v=None, attn_mask=None):
        """
        Args:
            x:      (B, T, C)       input embeddings
            past_k: (B, T_past, hs) cached keys, or None
            past_v: (B, T_past, hs) cached values, or None
        Returns:
            out:   (B, T, hs)           attention output
            new_k: (B, T_past+T, hs)    updated key cache   (None during training)
            new_v: (B, T_past+T, hs)    updated value cache  (None during training)
        """
        B, T, C = x.shape
        k = self.key(x)    # (B, T, hs)
        q = self.query(x)  # (B, T, hs)
        v = self.value(x)  # (B, T, hs)

        if not self.training:
            if past_k is not None:
                # ── Decode step: append new K/V onto cached past ──
                k = torch.cat([past_k, k], dim=1)  # (B, T_past + T, hs)
                v = torch.cat([past_v, v], dim=1)

                # Q attends over full cache — no causal mask needed (T=1)
                wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5

                if attn_mask is not None:
                    new_valid = torch.ones(B, 1, T, device=wei.device, dtype=torch.bool)
                    full_mask = torch.cat([attn_mask, new_valid], dim=-1)
                    wei = wei.masked_fill(~full_mask, float('-inf'))

                wei = F.softmax(wei, dim=-1)
                wei = self.dropout(wei)
                out = wei @ v
            else:
                # ── Prefill step: full prompt, needs causal mask ──
                wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
                wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
                wei = F.softmax(wei, dim=-1)
                wei = self.dropout(wei)
                out = wei @ v

            return out, k, v   # return updated cache
        else:
            # ── Training path — unchanged, no cache ──
            wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            wei = F.softmax(wei, dim=-1)
            wei = self.dropout(wei)
            out = wei @ v
            return out, None, None


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, attn_mask=None):
        """
        Args:
            x:       (B, T, C)
            past_kv: list of (past_k, past_v) per head, or None
        Returns:
            out:    (B, T, n_embd)
            new_kv: list of (new_k, new_v) per head
        """
        if past_kv is None:
            past_kv = [(None, None)] * len(self.heads)

        outputs, new_kvs = [], []
        for i, h in enumerate(self.heads):
            pk, pv = past_kv[i]
            out, nk, nv = h(x, pk, pv, attn_mask=attn_mask)
            outputs.append(out)
            new_kvs.append((nk, nv))

        out = torch.cat(outputs, dim=-1)
        out = self.dropout(self.proj(out))
        return out, new_kvs


class FeedFoward(nn.Module):
    """A simple linear layer followed by a non-linearity."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: communication followed by computation."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, past_kv=None, attn_mask=None):
        """
        Returns:
            x:      (B, T, n_embd)
            new_kv: list of (new_k, new_v) per head in this block
        """
        sa_out, new_kv = self.sa(self.ln1(x), past_kv, attn_mask=attn_mask)
        x = x + sa_out
        x = x + self.ffwd(self.ln2(x))
        return x, new_kv


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # ModuleList instead of Sequential so we can pass per-block cache
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, pos=None, past_kvs=None, attn_mask=None):
        """
        Args:
            idx:      (B, T) token indices
            targets:  (B, T) target indices, or None
            pos:      (B, T) explicit position indices, or None (uses arange)
            past_kvs: list-of-lists cache structure, or None
                      past_kvs[layer][head] = (key_tensor, value_tensor)
        Returns:
            logits:   (B, T, vocab_size)
            loss:     scalar or None
            new_kvs:  updated cache with same structure as past_kvs
        """
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)  # (B, T, C)

        if pos is None:
            pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, C)
        else:
            pos_emb = self.position_embedding_table(pos)  # (B, T, C)

        x = tok_emb + pos_emb  # (B, T, C)

        # Thread cache through each block
        if past_kvs is None:
            past_kvs = [None] * len(self.blocks)

        new_kvs = []
        for i, block in enumerate(self.blocks):
            x, block_kv = block(x, past_kvs[i], attn_mask=attn_mask)
            new_kvs.append(block_kv)

        x = self.ln_f(x)          # (B, T, C)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss, new_kvs

    def generate(self, idx, max_new_tokens):
        """Original generate (no cache, full recompute) for reference."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Training

Training is **unchanged** — during `model.train()`, every `Head` takes the training branch
and returns `(out, None, None)` for the cache. The cache is simply discarded via `_`.

In [6]:
model = GPTLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss, _ = model(xb, yb)  # _ discards the cache during training
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Quick sanity check with the original no-cache generate
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

0.209729 M parameters
step 0: train loss 4.1959, val loss 4.1962
step 100: train loss 2.6229, val loss 2.6166
step 200: train loss 2.4555, val loss 2.4488
step 300: train loss 2.3810, val loss 2.3928
step 400: train loss 2.3202, val loss 2.3223
step 500: train loss 2.2364, val loss 2.2541
step 600: train loss 2.1812, val loss 2.2234
step 700: train loss 2.1326, val loss 2.1583
step 800: train loss 2.0932, val loss 2.1352
step 900: train loss 2.0499, val loss 2.0987
step 1000: train loss 2.0349, val loss 2.0819
step 1100: train loss 1.9994, val loss 2.0706
step 1200: train loss 1.9857, val loss 2.0726
step 1300: train loss 1.9638, val loss 2.0401
step 1400: train loss 1.9386, val loss 2.0296
step 1500: train loss 1.9028, val loss 1.9947
step 1600: train loss 1.8821, val loss 1.9914
step 1700: train loss 1.8785, val loss 1.9813
step 1800: train loss 1.8746, val loss 1.9878
step 1900: train loss 1.8451, val loss 1.9599
step 2000: train loss 1.8294, val loss 1.9525
step 2100: train loss 1.

## What's Next: Hint 3 — The Scheduler Loop

Now that each request owns its own KV cache, the next step is the **scheduler loop**:

```
while there are active requests OR the waiting queue is non-empty:
    1. Check the waiting queue — can any new requests join the batch?
    2. Build the input tensor from ALL active requests (each contributes 1 token)
    3. Forward pass → get logits for all active requests at once
    4. Sample next token for each request
    5. Check: did any request hit max_new_tokens? → remove it, emit result
    6. Go to 1
```

The key challenge will be **padding the KV caches** to a common T dimension
when batching multiple requests (since they have different sequence lengths).
You'll need to `torch.cat` along dim=0 after padding along dim=1.

After un-batching the results, scatter the updated caches back to each request.

In [ ]:
def assemble_batch_cache(requests):
    """
    Gather per-request KV caches into batched tensors.
    LEFT-pads shorter caches so new tokens always land at the right edge.

    Big problem: You have 3 active requests. Each owns its own KV cache. You need to feed them to the model as one
    batched tensor. But their caches have different lengths:

    Returns:
        past_kvs:    batched cache structure  [layer][head] = (B, T_max, hs)
        attn_mask:   (B, 1, T_max) bool — True = valid, False = padding
        pad_lengths: list of int — how many pad positions per request (for disassembly)
    """

    B = len(requests)
    lengths = [req.kv_cache[(0, 0)][0].shape[1] for req in requests]
    max_t = max(lengths)

    pad_lengths = [max_t - t for t in lengths] # pad lengths for every position in t

    attn_mask = torch.zeros(B, 1, max_t, device=device, dtype=torch.bool)

    for i, pad in enumerate(pad_lengths):
        attn_mask[i, 0, pad:] = True

    past_kvs = []

    for layer_idx in range(n_layer):
        block_kv = []

        for head_idx in range(n_head):
            keys, values = [], []

            for i, req in enumerate(requests):
                k, v = req.kv_cache[(layer_idx, head_idx)]
                if pad_lengths[i] > 0:
                    hs = k.shape[2]
                    pad = torch.zeros(1, pad_lengths[i], hs, device=device)
                    k = torch.cat([pad, k], dim=1)
                    v = torch.cat([pad, v], dim=1)

                keys.append(k)
                values.append(v)

            block_kv.append((torch.cat(keys, dim=0), torch.cat(values, dim=0)))

        past_kvs.append(block_kv)

    return past_kvs, attn_mask, pad_lengths

def disassemble_batch_cache(requests, new_kvs, pad_lengths):
    """
    Scatter batched KV cache back to per-request storage.
    After Head's torch.cat, each row is (T_max + 1) — strip the left-padding.
    """
    for layer_idx, block_kv in enumerate(new_kvs):
        for head_idx, (batched_k, batched_v) in enumerate(block_kv):
            for i, req in enumerate(requests):
                pad = pad_lengths[i]
                req.kv_cache[(layer_idx, head_idx)] = (
                    batched_k[i : i + 1, pad:, :],      # (1, T_i + 1, hs)
                    batched_v[i : i + 1, pad:, :],
                )


def scheduled_generate(model, requests, policy="fcfs", token_budget=16, max_kv_tokens=256):
    scheduler = Scheduler(policy, token_budget=token_budget, max_kv_tokens=max_kv_tokens)

    step = 0

    for req in requests:
        req.arrival_time = step
        scheduler.add_request(req)
    
    model.eval()

    with torch.no_grad():
        while not scheduler.is_done():
            prefill_req, decode_reqs = scheduler.schedule(step)

            if prefill_req:
            
                prefill_chunk_tokens = []
                
                remaining_budget = token_budget - len(scheduler.active)

                if remaining_budget > 0 and scheduler.prefilling:
                    p_req = scheduler.prefilling[0]

                    tokens_left = len(p_req.prompt_tokens) - p_req.prefill_cursor
                    chunk_size = min(remaining_budget, tokens_left)

                    chunk_start = p_req.prefill_cursor 

                    chunk_tokens = p_req.prompt_tokens[chunk_start: chunk_start + chunk_size]

                    prefill_chunk_tokens = torch.tensor([chunk_tokens], dtype=torch.long, device=device)

                    p_req.prefill_cursor += chunk_size

                if len(prefill_chunk_tokens) == 0 and not scheduler.active:
                    step += 1
                    continue

                if len(prefill_chunk_tokens) > 0:
                    pos = torch.arange(chunk_start, chunk_start + chunk_size, device=device).unsqueeze(0)

                    if p_req.kv_cache:
                        #  This format is wrong 
                        # logits, _, new_kvs = model(prefill_chunk_tokens, past_kvs=req.kv_cache)
                        # list[list[(k, v)]] is shape
                        
                        past_kvs = []
                        for layer_idx in range(n_layer):
                            block_kv = [(p_req.kv_cache[(layer_idx, hi)]) for hi in range(n_head)] 
                            past_kvs.append(block_kv)
                        
                        logits, _, new_kvs = model(prefill_chunk_tokens, pos=pos, past_kvs=past_kvs)

                    else:
                        logits, _, new_kvs = model(prefill_chunk_tokens, pos=pos)

                    for li, bkv in enumerate(new_kvs):
                        for hi, (k, v) in enumerate(bkv):
                            p_req.kv_cache[(li, hi)] = (k, v)


                    logits = logits[:, -1, :]
                    probs = F.softmax(logits, dim=-1)
                    idx_next = torch.multinomial(probs, num_samples=1)

                    if prefill_req.is_fully_prefilled:
                        prefill_req.generated_tokens.append(idx_next.item())
                        prefill_req._last_token = idx_next
                        scheduler.promote(prefill_req)

            if decode_reqs:

                batch_tokens = torch.cat([req._last_token for req in scheduler.active])

                batch_positions = torch.tensor([[len(req.tokens_so_far) - 1] for req in scheduler.active], device=device)

                past_kvs, attn_mask, pad_lengths = assemble_batch_cache(scheduler.active)

                logits, _, new_kvs = model(
                    batch_tokens,
                    pos=batch_positions,
                    past_kvs=past_kvs,
                    attn_mask=attn_mask
                )

                logits = logits[:, -1, :]
                probs   = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)

                disassemble_batch_cache(scheduler.active, new_kvs, pad_lengths)

                for i, req in enumerate(scheduler.active):
                    req.generated_tokens.append(idx_next[i].item())
                    req._last_token = idx_next[i : i + 1]
                
                for req in list(scheduler.active):
                    if req.is_done:
                        scheduler.complete(req)
            
            step += 1
        
        return scheduler


In [43]:
# ══════════════════════════════════════════════════════════════
# Test 1: FCFS — same priority, all complete, valid output
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("Test 1: FCFS correctness")
print("=" * 60)
torch.manual_seed(42)
reqs = [
    Request(id=0, prompt_tokens=encode("O Romeo, "),     max_new_tokens=10),
    Request(id=1, prompt_tokens=encode("To be or "),     max_new_tokens=10),
    Request(id=2, prompt_tokens=encode("KING HENRY:\n"), max_new_tokens=10),
]
s = scheduled_generate(model, reqs, policy="fcfs", token_budget=16, max_kv_tokens=256)
for req in reqs:
    assert req.status == "done", f"❌ Req {req.id} not done"
    assert req.num_generated == 10, f"❌ Req {req.id}: got {req.num_generated} tokens, expected 10"
    print(f"  Req {req.id}: '{decode(req.tokens_so_far)}'")
print("✅ Test 1 passed")


Test 1: FCFS correctness
  Req 0: 'O Romeo, dings; nob'
  Req 1: 'To be or than thy's'
  Req 2: 'KING HENRY:
Hast nowes'
✅ Test 1 passed


In [44]:
# ══════════════════════════════════════════════════════════════
# Test 2: Same requests under FCFS vs Priority — admission order flips
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("Test 2: Priority jumping the queue")
print("=" * 60)

# ── FCFS: req 0 (lower id) admitted first ──
print("\n── FCFS run ──")
torch.manual_seed(42)
fcfs_reqs = [
    Request(id=0, prompt_tokens=encode("A " * 15), max_new_tokens=3, priority=2),
    Request(id=1, prompt_tokens=encode("B " * 3),  max_new_tokens=3, priority=0),
]
s_fcfs = scheduled_generate(model, fcfs_reqs, policy="fcfs", token_budget=16, max_kv_tokens=256)

# ── Priority: req 1 (priority=0) admitted first ──
print("\n── Priority run ──")
torch.manual_seed(42)
prio_reqs = [
    Request(id=0, prompt_tokens=encode("A " * 15), max_new_tokens=3, priority=2),
    Request(id=1, prompt_tokens=encode("B " * 3),  max_new_tokens=3, priority=0),
]
s_prio = scheduled_generate(model, prio_reqs, policy="priority", token_budget=16, max_kv_tokens=256)

for req in fcfs_reqs + prio_reqs:
    assert req.status == "done", f"❌ Req {req.id} not done"
    assert req.num_generated == 3

# Check step logs above:
#   FCFS     → [step 0] prefill=0  (req 0 admitted first, long prompt hogs budget)
#   Priority → [step 0] prefill=1  (req 1 jumps queue despite arriving at same time)
print("\n✅ Test 2 passed — verify from logs: FCFS admits req 0 first, Priority admits req 1 first")


Test 2: Priority jumping the queue

── FCFS run ──

── Priority run ──

✅ Test 2 passed — verify from logs: FCFS admits req 0 first, Priority admits req 1 first


In [45]:
# ══════════════════════════════════════════════════════════════
# Test 3: Tight KV budget forces preemption
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("Test 3: Preemption under memory pressure")
print("=" * 60)

torch.manual_seed(42)
reqs = [
    Request(id=0, prompt_tokens=encode("A " * 5), max_new_tokens=10, priority=2),  # low pri → victim
    Request(id=1, prompt_tokens=encode("B " * 5), max_new_tokens=5,  priority=0),  # high pri → stays
]
# 2 requests × 10-token prompt = 20 KV at admission. After a couple decode steps → exceeds 22.
s = scheduled_generate(model, reqs, policy="priority", token_budget=16, max_kv_tokens=22)

# Preemption must have fired
preempted_ids = [r.id for r in s.preempted]
assert len(s.preempted) > 0, "❌ No preemption occurred — lower max_kv_tokens"
assert 0 in preempted_ids, f"❌ Expected req 0 (low priority) to be preempted, got {preempted_ids}"

# Both requests still finish
for req in reqs:
    assert req.status == "done", f"❌ Req {req.id} stuck in status '{req.status}'"
    print(f"  Req {req.id}: {req.num_generated} tokens | '{decode(req.tokens_so_far)}'")

print(f"  Preempted IDs: {preempted_ids}")
print("✅ Test 3 passed")


Test 3: Preemption under memory pressure
  Req 0: 10 tokens | 'A A A A A cCafpier,-'
  Req 1: 5 tokens | 'B B B B B Plunk'
  Preempted IDs: [0]
✅ Test 3 passed


In [49]:
# ══════════════════════════════════════════════════════════════
# Test 4: After preemption + re-prefill, KV cache is consistent
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("Test 4: Preempted request re-enters correctly")
print("=" * 60)

# ── Baseline: req 0 alone, no preemption ──
torch.manual_seed(42)
baseline = Request(id=0, prompt_tokens=encode("A " * 5), max_new_tokens=10)
scheduled_generate(model, [baseline], policy="fcfs", token_budget=16, max_kv_tokens=256)

# ── Stress: req 0 gets preempted, must re-prefill ──
torch.manual_seed(42)
req0 = Request(id=0, prompt_tokens=encode("A " * 5), max_new_tokens=10, priority=2)
req1 = Request(id=1, prompt_tokens=encode("B " * 5), max_new_tokens=5,  priority=0)
s = scheduled_generate(model, [req0, req1], policy="priority", token_budget=16, max_kv_tokens=22)

assert req0.status == "done"
assert req0.num_generated == req0.max_new_tokens

# KEY CHECK: KV cache length must equal prompt + generated tokens.
# If generated_tokens wasn't reset on preemption, old tokens have no KV backing
# and this assertion will fail — exposing the bug.
expected_kv_len = len(req0.prompt_tokens) + req0.num_generated - 1
actual_kv_len = req0.kv_cache[(0, 0)][0].shape[1]
assert actual_kv_len == expected_kv_len, \
    f"❌ KV cache has {actual_kv_len} entries, expected {expected_kv_len}. " \
    f"Hint: generated_tokens should be reset to [] on preemption."

print(f"  Baseline:  '{decode(baseline.tokens_so_far)}'")
print(f"  Stressed:  '{decode(req0.tokens_so_far)}'")
print(f"  KV cache length: {actual_kv_len} ✓")
print("✅ Test 4 passed")


Test 4: Preempted request re-enters correctly
  Baseline:  'A A A A A drie as fa'
  Stressed:  'A A A A A Cafpter,-''
  KV cache length: 19 ✓
✅ Test 4 passed
